<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 4


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Описание задачи: 
Создать базовый класс Product в C#, который будет представлять информацию о 
продуктах.  На  основе  этого  класса  разработать  2-3  производных  класса, 
демонстрирующих принципы наследования и полиморфизма. В каждом из классов 
должны быть реализованы новые атрибуты и методы, а также переопределены 
некоторые методы базового класса для демонстрации полиморфизма. 
Требования к базовому классу Product: 

• Атрибуты: Название (Name), Цена (Price), Производитель (Manufacturer). 

• Методы: 

    o GetInfo(): метод для получения информации о продукте в виде строки. 

    o Discount(): метод для применения скидки к цене продукта.

    o Display(): метод для отображения информации о продукте на экране.
     
Требования к производным классам: 
1. Электроника  (Electronics):  Должен  содержать  дополнительные  атрибуты, 
такие как Гарантийный срок (WarrantyPeriod). Метод Discount() должен быть 
переопределен  для  добавления  логики  учета  гарантийного  срока  при 
применении скидки. 
2. Одежда (Clothing): Должен содержать дополнительные атрибуты, такие как 
Размер (Size). Метод Display() должен быть переопределен для добавления 
информации о размере при отображении информации о продукте. 
3. Книги  (Books) (если  требуется  третий  класс):  Должен  содержать 
дополнительные атрибуты, такие как Автор (Author). Метод GetInfo() должен 
быть  переопределен  для  включения  информации  об  авторе  в  описании 
продукта

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface IDiscountable 
{ 
    void ApplyDiscount(decimal percentage);
    void ApplyDiscount(decimal percentage, DateTime expiryDate); // Перегрузка метода
}

public interface IShippable 
{ 
    decimal CalculateShipping();
    decimal CalculateShipping(string shippingType); // Перегрузка метода
}

public interface IRatable
{
    void AddRating(int rating);
    double GetAverageRating();
    int GetRatingCount();
}

// Generic класс для работы с коллекциями
public class ProductCollection<T> where T : Product
{
    private List<T> products = new List<T>();

    public void Add(T product) => products.Add(product);
    
    public void Remove(T product) => products.Remove(product);
    
    public T FindByName(string name) => products.FirstOrDefault(p => p.Name.Equals(name, StringComparison.OrdinalIgnoreCase));
    
    public IEnumerable<T> GetByManufacturer(string manufacturer) => 
        products.Where(p => p.Manufacturer.Equals(manufacturer, StringComparison.OrdinalIgnoreCase));
    
    public IEnumerable<T> GetInStock() => products.Where(p => p.StockQuantity > 0);
    
    public IEnumerable<T> SortByPrice(bool ascending = true) => 
        ascending ? products.OrderBy(p => p.Price) : products.OrderByDescending(p => p.Price);
    
    public int Count => products.Count;
    
    public void DisplayAll()
    {
        Console.WriteLine($"📦 Коллекция продуктов ({Count} items):");
        foreach (var product in products)
        {
            Console.WriteLine(product.GetBasicInfo());
        }
    }
}

public abstract class Product : IDiscountable, IRatable
{
    public string Name { get; set; }
    public decimal Price { get; set; }
    public string Manufacturer { get; set; }
    public int StockQuantity { get; set; }
    public string Description { get; set; } // Новый атрибут
    public string SKU { get; set; } // Новый атрибут
    public DateTime CreatedDate { get; set; } // Новый атрибут
    public bool IsAvailable { get; set; } // Новый атрибут
    
    private List<int> ratings = new List<int>(); // Новый атрибут для рейтингов

    public Product(string name, decimal price, string manufacturer, int stock, string description = "", string sku = "")
    {
        Name = name;
        Price = price;
        Manufacturer = manufacturer;
        StockQuantity = stock;
        Description = description;
        SKU = string.IsNullOrEmpty(sku) ? GenerateSKU() : sku;
        CreatedDate = DateTime.Now;
        IsAvailable = stock > 0;
    }

    // Новые методы
    public virtual string GetBasicInfo() => $"{Name} - {Price:C} ({Manufacturer})";
    
    public virtual void UpdateStock(int quantity)
    {
        StockQuantity += quantity;
        IsAvailable = StockQuantity > 0;
        Console.WriteLine($"Запас {Name} обновлен: {StockQuantity} шт.");
    }
    
    public virtual bool CanPurchase(int quantity = 1) => IsAvailable && StockQuantity >= quantity;
    
    public virtual void Purchase(int quantity = 1)
    {
        if (CanPurchase(quantity))
        {
            StockQuantity -= quantity;
            IsAvailable = StockQuantity > 0;
            Console.WriteLine($"Покупка: {quantity} x {Name}");
        }
        else
        {
            Console.WriteLine($"Недостаточно запаса для {Name}");
        }
    }

    // Перегрузка метода ApplyDiscount
    public virtual void ApplyDiscount(decimal percentage) 
    {
        decimal oldPrice = Price;
        Price -= Price * (percentage / 100);
        Console.WriteLine($"Скидка {percentage}% применена к {Name}: {oldPrice:C} -> {Price:C}");
    }
    
    public virtual void ApplyDiscount(decimal percentage, DateTime expiryDate)
    {
        if (DateTime.Now <= expiryDate)
        {
            ApplyDiscount(percentage);
            Console.WriteLine($"Скидка действительна до: {expiryDate:dd.MM.yyyy}");
        }
        else
        {
            Console.WriteLine($"Скидка для {Name} истекла");
        }
    }

    // Реализация IRatable
    public void AddRating(int rating)
    {
        if (rating >= 1 && rating <= 5)
        {
            ratings.Add(rating);
            Console.WriteLine($"Рейтинг {rating}/5 добавлен для {Name}");
        }
        else
        {
            Console.WriteLine("Рейтинг должен быть от 1 до 5");
        }
    }
    
    public double GetAverageRating() => ratings.Count > 0 ? ratings.Average() : 0;
    
    public int GetRatingCount() => ratings.Count;

    public virtual string GetInfo() => 
        $"Название: {Name}\nЦена: {Price:C}\nПроизводитель: {Manufacturer}\nОписание: {Description}\nSKU: {SKU}\nВ наличии: {StockQuantity}";

    public abstract string GetCategory();
    
    protected virtual string GenerateSKU() => $"{Manufacturer.Substring(0, Math.Min(3, Manufacturer.Length))}-{Name.Substring(0, Math.Min(5, Name.Length))}-{DateTime.Now:MMdd}";
}

public class Electronics : Product, IShippable
{
    public int WarrantyPeriod { get; set; }
    public double Weight { get; set; } // Новый атрибут
    public string PowerRequirements { get; set; } // Новый атрибут
    public bool HasBattery { get; set; } // Новый атрибут
    
    public Electronics(string name, decimal price, string manufacturer, int stock, int warranty, 
                      double weight = 0, string power = "100-240V", bool hasBattery = false, string description = "") 
        : base(name, price, manufacturer, stock, description)
    {
        WarrantyPeriod = warranty;
        Weight = weight;
        PowerRequirements = power;
        HasBattery = hasBattery;
    }

    // Убрано sealed чтобы позволить переопределение в производных классах
    public override string GetInfo() => 
        base.GetInfo() + $"\nГарантия: {WarrantyPeriod} мес.\nВес: {Weight} кг\nПитание: {PowerRequirements}\nБатарея: {(HasBattery ? "Да" : "Нет")}";

    // Новые методы
    public double CalculateWeightCost() => Weight * 50; // 50 руб/кг
    
    public virtual string GetWarrantyInfo() => $"Гарантия производителя: {WarrantyPeriod} месяцев";
    
    // Перегрузка метода CalculateShipping
    public decimal CalculateShipping() => 300 + (decimal)CalculateWeightCost();
    
    public decimal CalculateShipping(string shippingType)
    {
        return shippingType.ToLower() switch
        {
            "express" => CalculateShipping() * 2,
            "pickup" => 0,
            "international" => CalculateShipping() * 3,
            _ => CalculateShipping()
        };
    }

    public override string GetCategory() => "Электроника";
}

public class Clothing : Product, IShippable
{
    public string Size { get; set; }
    public string Color { get; set; } // Новый атрибут
    public string Material { get; set; } // Новый атрибут
    public string CareInstructions { get; set; } // Новый атрибут
    
    public Clothing(string name, decimal price, string manufacturer, int stock, string size, 
                   string color = "Разноцветный", string material = "Хлопок", string care = "Машинная стирка", string description = "") 
        : base(name, price, manufacturer, stock, description)
    {
        Size = size;
        Color = color;
        Material = material;
        CareInstructions = care;
    }

    // Убрано sealed
    public override string GetInfo() => 
        base.GetInfo() + $"\nРазмер: {Size}\nЦвет: {Color}\nМатериал: {Material}\nУход: {CareInstructions}";

    // Новые методы
    public virtual string GetSizeGuide() => $"Руководство по размерам для {Manufacturer}: {Size}";
    
    public bool IsSizeAvailable(string requestedSize) => 
        Size.Equals(requestedSize, StringComparison.OrdinalIgnoreCase) && IsAvailable;

    // Перегрузка метода CalculateShipping
    public decimal CalculateShipping() => 150;
    
    public decimal CalculateShipping(string shippingType)
    {
        return shippingType.ToLower() switch
        {
            "express" => 300,
            "pickup" => 0,
            "international" => 500,
            _ => 150
        };
    }

    public override string GetCategory() => "Одежда";
}

public class Smartphone : Electronics
{
    public string OS { get; set; }
    public int Storage { get; set; } // Новый атрибут (GB)
    public string Camera { get; set; } // Новый атрибут
    public bool Has5G { get; set; } // Новый атрибут
    
    public Smartphone(string name, decimal price, string manufacturer, int stock, int warranty, string os, 
                     int storage = 64, string camera = "12MP", bool has5G = true, double weight = 0.2, string description = "") 
        : base(name, price, manufacturer, stock, warranty, weight, "5V", true, description)
    {
        OS = os;
        Storage = storage;
        Camera = camera;
        Has5G = has5G;
    }

    // Теперь можно переопределить GetInfo(), так как убрано sealed
    public override string GetInfo() => 
        base.GetInfo() + $"\nОС: {OS}\nПамять: {Storage}GB\nКамера: {Camera}\n5G: {(Has5G ? "Да" : "Нет")}";

    // Перегрузка метода ApplyDiscount
    public override void ApplyDiscount(decimal percentage)
    {
        if (percentage > 20)
        {
            Console.WriteLine($"Максимальная скидка для смартфонов - 20%. Установлено 20% вместо {percentage}%");
            percentage = 20;
        }
        base.ApplyDiscount(percentage);
    }

    // Новые методы
    public string GetTechSpecs() => $"Память: {Storage}GB, Камера: {Camera}, 5G: {Has5G}";
    
    public override string GetWarrantyInfo() => 
        $"Расширенная гарантия на смартфон: {WarrantyPeriod} месяцев + 3 месяца бесплатной поддержки";
}

// Новый производный класс
public class Book : Product
{
    public string Author { get; set; }
    public string ISBN { get; set; }
    public int PageCount { get; set; }
    public string Genre { get; set; }
    
    public Book(string name, decimal price, string manufacturer, int stock, string author, string isbn, int pages, string genre, string description = "") 
        : base(name, price, manufacturer, stock, description)
    {
        Author = author;
        ISBN = isbn;
        PageCount = pages;
        Genre = genre;
    }

    public override string GetInfo() => 
        base.GetInfo() + $"\nАвтор: {Author}\nISBN: {ISBN}\nСтраниц: {PageCount}\nЖанр: {Genre}";

    // Перегрузка метода ApplyDiscount
    public override void ApplyDiscount(decimal percentage)
    {
        if (percentage > 30)
        {
            Console.WriteLine($"Максимальная скидка для книг - 30%. Установлено 30% вместо {percentage}%");
            percentage = 30;
        }
        base.ApplyDiscount(percentage);
    }

    public override string GetCategory() => "Книги";
    
    // Новый метод
    public string GetReadingTime() => 
        PageCount < 100 ? "Быстрое чтение" : PageCount < 300 ? "Среднее время чтения" : "Длительное чтение";
}

public class ShoppingCart
{
    private List<Product> items = new List<Product>();

    public void AddProduct(Product product) 
    {
        if (product.CanPurchase())
        {
            items.Add(product);
            Console.WriteLine($"✅ {product.Name} добавлен в корзину");
        }
        else
        {
            Console.WriteLine($"❌ {product.Name} нет в наличии");
        }
    }
    
    public void RemoveProduct(Product product) => items.Remove(product);
    
    public void ShowCart()
    {
        Console.WriteLine("\n🛒 Корзина:");
        foreach (var item in items) 
        {
            Console.WriteLine(item.GetBasicInfo());
            Console.WriteLine("---");
        }
    }
    
    public decimal GetTotal()
    {
        decimal total = items.Sum(item => item.Price);
        Console.WriteLine($"$ Итого: {total:C}");
        return total;
    }
    
    public void Checkout()
    {
        Console.WriteLine("\n💳 Оформление заказа...");
        foreach (var item in items)
        {
            item.Purchase();
        }
        Console.WriteLine($"Заказ оформлен! Итого: {GetTotal():C}");
        items.Clear();
    }
}

public class ProgramDemo
{
    public static void RunDemo()
    {
        Console.WriteLine("🛍️ СИСТЕМА МАГАЗИНА \n");

        // Создание продуктов
        Smartphone phone = new Smartphone("iPhone 15", 89990, "Apple", 10, 24, "iOS", 128, "48MP", true, 0.17);
        Clothing jacket = new Clothing("Куртка", 7500, "Nike", 20, "L", "Черный", "Полиэстер", "Химчистка");
        Book book = new Book("Война и мир", 1200, "Эксмо", 15, "Лев Толстой", "978-5-699-12001-2", 1225, "Классика");
        
        // Демонстрация полиморфизма
        Console.WriteLine("=== ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА ===");
        Product[] products = { phone, jacket, book };
        
        foreach (var product in products)
        {
            Console.WriteLine($"\n{product.GetCategory()}: {product.Name}");
            Console.WriteLine(product.GetInfo());
            
            // Полиморфизм через перекрытие методов
            product.ApplyDiscount(15);
            
            // Демонстрация интерфейсов
            if (product is IShippable shippable)
            {
                Console.WriteLine($"Доставка: {shippable.CalculateShipping():C}");
                Console.WriteLine($"Экспресс доставка: {shippable.CalculateShipping("express"):C}");
            }
            
            // Добавление рейтингов
            product.AddRating(5);
            product.AddRating(4);
            Console.WriteLine($"Рейтинг: {product.GetAverageRating():F1} ({product.GetRatingCount()} отзывов)");
        }

        // Демонстрация перегрузки методов
        Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ ПЕРЕГРУЗКИ МЕТОДОВ ===");
        phone.ApplyDiscount(25, DateTime.Now.AddDays(7)); // Скидка с expiry date
        book.ApplyDiscount(40); // Проверка ограничения скидки

        // Демонстрация generic класса
        Console.WriteLine("\n=== ДЕМОНСТРАЦИЯ GENERIC КЛАССА ===");
        ProductCollection<Electronics> electronicsCollection = new ProductCollection<Electronics>();
        electronicsCollection.Add(phone);
        electronicsCollection.Add(new Electronics("Ноутбук", 55000, "Lenovo", 5, 12, 2.5));
        
        electronicsCollection.DisplayAll();
        Console.WriteLine($"В наличии: {electronicsCollection.GetInStock().Count()} товаров");

        // Работа с корзиной
        Console.WriteLine("\n=== РАБОТА С КОРЗИНОЙ ===");
        ShoppingCart cart = new ShoppingCart();
        cart.AddProduct(phone);
        cart.AddProduct(jacket);
        cart.AddProduct(book);
        
        cart.ShowCart();
        cart.GetTotal();
        
        // Демонстрация покупки
        cart.Checkout();
        
        // Проверка обновления запаса
        Console.WriteLine($"\nЗапас iPhone после покупки: {phone.StockQuantity}");
        Console.WriteLine($"Доступен для покупки: {phone.CanPurchase()}");
    }
}

ProgramDemo.RunDemo();

🛍️ СИСТЕМА МАГАЗИНА 

=== ДЕМОНСТРАЦИЯ ПОЛИМОРФИЗМА ===

Электроника: iPhone 15
Название: iPhone 15
Цена: ¤89,990.00
Производитель: Apple
Описание: 
SKU: App-iPhon-1017
В наличии: 10
Гарантия: 24 мес.
Вес: 0.17 кг
Питание: 5V
Батарея: Да
ОС: iOS
Память: 128GB
Камера: 48MP
5G: Да
Скидка 15% применена к iPhone 15: ¤89,990.00 -> ¤76,491.50
Доставка: ¤308.50
Экспресс доставка: ¤617.00
Рейтинг 5/5 добавлен для iPhone 15
Рейтинг 4/5 добавлен для iPhone 15
Рейтинг: 4.5 (2 отзывов)

Одежда: Куртка
Название: Куртка
Цена: ¤7,500.00
Производитель: Nike
Описание: 
SKU: Nik-Куртк-1017
В наличии: 20
Размер: L
Цвет: Черный
Материал: Полиэстер
Уход: Химчистка
Скидка 15% применена к Куртка: ¤7,500.00 -> ¤6,375.00
Доставка: ¤150.00
Экспресс доставка: ¤300.00
Рейтинг 5/5 добавлен для Куртка
Рейтинг 4/5 добавлен для Куртка
Рейтинг: 4.5 (2 отзывов)

Книги: Война и мир
Название: Война и мир
Цена: ¤1,200.00
Производитель: Эксмо
Описание: 
SKU: Экс-Война-1017
В наличии: 15
Автор: Лев Толстой
ISBN: 978-5-699-1